# Project Description
This is under construction and will be updated soon

## Initial Setup

- Setup **wandb** (weights and biases) to record and track the fine tuning performance and the results of benchmarking across different runs. Create a project on **wandb**, add API key and initialize wandb
- Load `Qwen3.5-4B` from **unsloth** and initialize the weights
- Load the dataset `lavita/MedQuad` from **HuggingFace**

In [1]:
import os
import wandb
from dotenv import load_dotenv

load_dotenv()

# Initialize wandb to track performance
wandb.init(
    project=os.environ["WANDB_PROJECT"],
    name="qwen-4b-medquad-run1",
    tags=["qwen", "16-bit", "medquad"]
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: loknezmonzter (loknezmonzter-dev) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
from unsloth import FastVisionModel

# Initialize model weights and tokenizer
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/qwen3.5-4B",
    load_in_4bit=False,
    use_gradient_checkpointing=True
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


weave: weave version 0.52.38 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
[weave.trace.init_message|INFO]weave version 0.52.38 is available!  To upgrade, please run:
 $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: loknezmonzter.
weave: View Weave data at https://wandb.ai/loknezmonzter-dev/finetuning_medquad/weave
[weave.trace.init_message|INFO]Logged in as Weights & Biases user: loknezmonzter.
View Weave data at https://wandb.ai/loknezmonzter-dev/finetuning_medquad/weave


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A4000. Num GPUs = 1. Max memory: 14.615 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu129. CUDA: 8.6. CUDA Toolkit: 12.9. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [8]:
from datasets import load_dataset

medquad_dataset = load_dataset("lavita/MedQuAD")
medquad_dataset = medquad_dataset["train"]
medquad_dataset, medquad_dataset[0]

(Dataset({
     features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
     num_rows: 47441
 }),
 {'document_id': '0000559',
  'document_source': 'GHR',
  'document_url': 'https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair',
  'category': None,
  'umls_cui': 'C0343073',
  'umls_semantic_types': 'T047',
  'umls_semantic_group': 'Disorders',
  'synonyms': 'KWWH',
  'question_id': '0000559-1',
  'question_focus': 'keratoderma with woolly hair',
  'question_type': 'information',
  'question': 'What is (are) keratoderma with woolly hair ?',
  'answer': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. 

In [12]:
# Split the dataset - 90% for training and 10% for testing
split_dataset = medquad_dataset.train_test_split(test_size=0.1, seed=42)
split_dataset

DatasetDict({
    train: Dataset({
        features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
        num_rows: 42696
    })
    test: Dataset({
        features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
        num_rows: 4745
    })
})

## Dataset Processing - Prepare For Fine Tuning

#### Prompt Formatting
Build the data dictionary so that every fine tuning prompt passed to LLM is in the format expected by it (**ChatML**). The chat template is applied to the tokenizer so that the tokenizer object is permenantly modified. This allows the tokenizer to know how to use Qwen specific tags `<|im_start|>` and `<|im_end|>`

The ChatML template must be bound to the tokenizer first before mapping the dataset.

#### Train On Responses Only
This is a technique that ensures **Completion-Only Training** for the benchmarking protocol. It ensures the model's loss function only optimizes for generating the medical answer rather than wasting computational power learning how to memorize the user's questions

#### Dataset Split
The dataset needs to be split into 90-10 ratio where 90% is split for training and the remaining 10% for testing. The data processing steps need to be applied to both the train and test splits. If the processing is only applied on the train split, the `SFTTrainer` will be looking for specific ChatML tags in the test set when calculating the **validation loss** and since the tags are not present, incorrect validation loss will be reported 

In [13]:
train_split, test_split

(Dataset({
     features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
     num_rows: 42696
 }),
 Dataset({
     features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
     num_rows: 4745
 }))

## Fine Tuning Setup

- Configure model for fine tuning
- Update the tokenizer with Qwen chat template
- 